# Camelyon17-WILDS: CNN baseline training

This notebook trains a CNN as a baseline. It loads Camelyon17-WILDS with the official WILDS dataset API and uses the benchmark splits: `train` for fitting, `val` for out-of-distribution (OOD) validation, `id_val` for an in-distribution comparison, and `test` for final evaluation.

Run from the project root. This notebook assumes the WILDS package is installed (`pip install wilds`) and the dataset is already present under `data/`.


In [ ]:
# If needed, uncomment and run in your environment.
# %pip install wilds torch torchvision scikit-learn matplotlib


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from wilds import get_dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
# Prefer GPU 3 when available; otherwise use GPU 0 or CPU.
if torch.cuda.is_available():
    DEVICE = torch.device('cuda:3' if torch.cuda.device_count() > 3 else 'cuda:0')
else:
    DEVICE = torch.device('cpu')
print('Device:', DEVICE)


In [ ]:
# Load the locally available dataset and its official WILDS splits.
dataset = get_dataset(
    dataset='camelyon17',
    root_dir='data',
    download=False,
)
print(dataset)


## Configure the baseline

There are no random flips, rotations, color changes, or other augmentations in this notebook. Images are only resized and normalized. `small_cnn` is a compact model suitable for a first run; set `MODEL_NAME` to `resnet18` for a second architecture.


In [ ]:
MODEL_NAME = 'small_cnn'    # 'small_cnn' or 'resnet18'
IMAGE_SIZE = 96
BATCH_SIZE = 128
EPOCHS = 8
LEARNING_RATE = 1e-3
NUM_WORKERS = 0            # try 2-4 on Unix/GPU if data loading is a bottleneck

# No augmentation: the same deterministic transform is used for every split.
image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3),
])

train_data = dataset.get_subset('train', transform=image_transform)
val_data = dataset.get_subset('val', transform=image_transform)       # OOD validation hospital
id_val_data = dataset.get_subset('id_val', transform=image_transform) # in-distribution validation
test_data = dataset.get_subset('test', transform=image_transform)     # held-out test hospital

print(f'Train: {len(train_data):,} | OOD validation: {len(val_data):,} | ID validation: {len(id_val_data):,} | Test: {len(test_data):,}')


In [ ]:
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=DEVICE.type=='cuda')
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type=='cuda')
id_val_loader = DataLoader(id_val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type=='cuda')
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type=='cuda')

# WILDS batches contain (image, label, metadata). Visualize one training patch.
images, labels, metadata_batch = next(iter(train_loader))
img = (images[0].permute(1,2,0).numpy() * 0.5 + 0.5).clip(0,1)
plt.figure(figsize=(3,3)); plt.imshow(img); plt.title(f'tumor={int(labels[0].item())}'); plt.axis('off');


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128,192,3,padding=1), nn.BatchNorm2d(192), nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(192, 2)
    def forward(self, x): return self.classifier(self.features(x).flatten(1))

def make_model(name):
    if name == 'small_cnn': return SmallCNN()
    if name == 'resnet18':
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, 2)
        return model
    raise ValueError("MODEL_NAME must be 'small_cnn' or 'resnet18'")

model = make_model(MODEL_NAME).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
print('Model:', MODEL_NAME)


In [ ]:
def unpack_batch(batch):
    # WILDS batches are (images, labels, metadata); metadata is not a model input.
    images, labels = batch[0], batch[1]
    return images, labels.reshape(-1).long()

def predict(model, loader):
    model.eval(); ys=[]; probs=[]
    with torch.no_grad():
        for batch in loader:
            x,y=unpack_batch(batch)
            logits=model(x.to(DEVICE, non_blocking=True))
            probs.extend(torch.softmax(logits, dim=1)[:,1].cpu().numpy())
            ys.extend(y.numpy())
    return np.asarray(ys), np.asarray(probs)

def metrics(y,p):
    pred=(p>=0.5).astype(int)
    return {
        'accuracy': accuracy_score(y,pred),
        'balanced_accuracy': balanced_accuracy_score(y,pred),
        'roc_auc': roc_auc_score(y,p) if len(np.unique(y))==2 else float('nan'),
    }


In [ ]:
history=[]
best_val_bacc=-1.0
checkpoint_path=f'best_{MODEL_NAME}_no_augmentation.pt'
for epoch in range(1,EPOCHS+1):
    model.train(); running_loss=0.0; seen=0
    for batch in train_loader:
        x,y=unpack_batch(batch)
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss=criterion(model(x),y); loss.backward(); optimizer.step()
        running_loss += loss.item()*len(y); seen += len(y)
    yv,pv=predict(model,val_loader); vm=metrics(yv,pv)
    row={'epoch':epoch,'train_loss':running_loss/seen,**{f'ood_val_{k}':v for k,v in vm.items()}}
    history.append(row)
    print(f"Epoch {epoch:02d}/{EPOCHS} | loss {row['train_loss']:.4f} | OOD val acc {vm['accuracy']:.4f} | balanced acc {vm['balanced_accuracy']:.4f} | AUC {vm['roc_auc']:.4f}")
    if vm['balanced_accuracy']>best_val_bacc:
        best_val_bacc=vm['balanced_accuracy']
        torch.save({'model_state_dict':model.state_dict(),'epoch':epoch,'val_metrics':vm,'model_name':MODEL_NAME},checkpoint_path)

history_df=__import__('pandas').DataFrame(history)
display(history_df)


## Evaluate the selected checkpoint

The checkpoint is chosen by OOD validation balanced accuracy. The test set is evaluated after training as a final held-out estimate. Avoid using test results to tune hyperparameters.


In [ ]:
checkpoint=torch.load(checkpoint_path,map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
print('Selected epoch:',checkpoint['epoch'],'| OOD validation:',checkpoint['val_metrics'])

results=[]
for split_name,loader in [('OOD validation',val_loader),('ID validation',id_val_loader),('held-out test',test_loader)]:
    y,p=predict(model,loader); m=metrics(y,p)
    results.append({'split':split_name,**m,'n':len(y)})
    print(split_name,m)
import pandas as pd
display(pd.DataFrame(results))


In [ ]:
y_test,p_test=predict(model,test_loader)
pred_test=(p_test>=0.5).astype(int)
ConfusionMatrixDisplay(confusion_matrix(y_test,pred_test),display_labels=['no tumor','tumor']).plot(cmap='Blues',values_format='d')
plt.title('Held-out test hospital'); plt.show()


## Notes for the report

Record the model, seed, epoch count, and metrics. Compare OOD validation with ID validation to describe the hospital distribution shift. This notebook provides the **no-augmentation baseline**; make a separate run or notebook with training augmentation enabled for the project comparison. Results are patch-level research measurements, not clinical advice.

The dataset and splits are provided by [WILDS](https://github.com/p-lambda/wilds) (Koh et al., *WILDS: A Benchmark of in-the-Wild Distribution Shifts*, ICML 2021). Cite WILDS and any methods or code you use in the report.
